# Survey speed vs. image quality: static PNG

Matplotlib version of the figure built in `04_interactive.ipynb`, from the same cached tables.
6 x 4 in, 12 pt axis labels, `TRANSPARENT` background switch (set to white while iterating; flip
back to transparent for delivery), Rubin colour palette (`ohlp/style.py`).

`PERIOD = "early_ops"` (default) draws a **single summary point** for Early Ops, 24 Oct–30 Nov 2025,
i.e. the nights before the Dec 2025–Mar 2026 sequence of downtimes (water condensation, Dynalene loss,
vacuum leak, earthquake; see `nightly_survey_speed/plots/downtime_summary.md`), with its nights as context.
The other values reuse the tables written by `01_data_and_binning.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

import pandas as pd

warnings.simplefilter("ignore", FutureWarning)
sys.path.insert(0, ".")
from ohlp import binning, data, static, reference as ref

In [ ]:
# Parameters (must match 01_data_and_binning.ipynb)
DAY_OBS_MIN = ref.DAY_OBS_EARLY_OPS_START
DAY_OBS_MAX = 20261014
PERIOD = "early_ops"  # "early_ops" (single Oct+Nov 2025 point), "window", "quarter" or "month"
IQ_COL = "idiq_aos_cam"
FIGSIZE = (6, 4)  # inches
LABEL_SIZE = 12  # pt, axis labels
TRANSPARENT = True  # False = white background (easier on a dark screen); True for delivery
TITLE = "Early Ops, 24 Oct – 30 Nov 2025"  # None for no title

# Single Early Ops window: first survey night to the last night before the downtime sequence.
EARLY_OPS_WINDOW = {"Early Ops, Oct–Nov 2025": (ref.DAY_OBS_EARLY_OPS_START, 20251130)}

tag = f"{DAY_OBS_MIN}_{DAY_OBS_MAX}"
nightly = pd.read_parquet(data.DATA_DIR / f"nightly_{tag}.parquet")
if PERIOD == "early_ops":
    (lo, hi), = EARLY_OPS_WINDOW.values()
    visits = pd.read_parquet(data.DATA_DIR / f"visits_{tag}.parquet")
    periods = binning.window_table(visits, nightly, EARLY_OPS_WINDOW)
    stem = f"early_ops_octnov_{lo}-{hi}"
    periods.to_parquet(data.DATA_DIR / f"window_early_ops_octnov_{lo}_{hi}.parquet")
else:
    TABLE_NAME = {"window": "windows", "quarter": "quarterly", "month": "monthly"}[PERIOD]
    periods = pd.read_parquet(data.DATA_DIR / f"{TABLE_NAME}_{tag}.parquet")
    stem = f"{TABLE_NAME}_{DAY_OBS_MIN}-{DAY_OBS_MAX}"
periods.query("series == 'all'")[["label_long", "n_nights_calendar", "n_nights_with_visits", "n_visits", "n_vis_expected", "ratio", f"{IQ_COL}_med"]].round(3)

In [ ]:
fig, ax = static.build_static_figure(
    periods,
    nightly,
    iq_col=IQ_COL,
    figsize=FIGSIZE,
    label_size=LABEL_SIZE,
    transparent=TRANSPARENT,
    title=TITLE,
)
out = static.save_png(
    fig,
    data.DATA_DIR.parent / "figures" / f"speed_vs_iq_static_{stem}.png",
    transparent=TRANSPARENT,
)
print(out)